In [1]:
%pip install \
    "qiskit==2.2.3" \
    "qiskit-machine-learning==0.9.0" \
    "qiskit-optimization==0.7.0" \
    "qiskit-algorithms==0.4.0" \
    "packaging>=24.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 11.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.7 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=f83cc776bd5f4c507b5d3285564e56b0b1154d8511ff75d842b77e453eb9b112
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [12]:
import numpy as np
from importlib.metadata import version
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('Qiskit:', version('qiskit'))
from qiskit.quantum_info import partial_trace, purity

data = np.array([0.2, 0.6, 0.8])
qc = QuantumCircuit(3)
for i, value in enumerate(data):
    qc.ry(float(np.pi * value), i)
before = Statevector.from_instruction(qc)
qc.cx(0, 1)
qc.cx(1, 2)
after = Statevector.from_instruction(qc)
print('얽힘 전 확률:', np.round(before.probabilities(), 6))
print('얽힘 후 확률:', np.round(after.probabilities(), 6))
reduced = partial_trace(after, [1, 2])
local_purity = float(purity(reduced).real)
print('0번 큐비트 축약상태 순도:', round(local_purity, 6))
assert np.isclose(after.probabilities().sum(), 1.0)
assert local_purity < 1 - 1e-8

Qiskit: 2.2.3
얽힘 전 확률: [0.029841 0.00315  0.056532 0.005968 0.282659 0.029841 0.535477 0.056532]
얽힘 후 확률: [0.029841 0.005968 0.535477 0.029841 0.282659 0.056532 0.056532 0.00315 ]
0번 큐비트 축약상태 순도: 0.983504


CX는 입력 상태에 따라 얽힘을 만듭니다. 확률 변화와 함께 순도를 출력하여 얽힘 여부를 수치로 확인합니다.

In [11]:
import numpy as np
from importlib.metadata import version
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('Qiskit:', version('qiskit'))
from qiskit.circuit import ParameterVector

x = ParameterVector('x', 3)
theta = ParameterVector('theta', 3)  # 세타: 학습 가능한 추가 회전각
qc = QuantumCircuit(3)
for i in range(3):
    qc.ry(np.pi * x[i] + theta[i], i)
qc.cx(0, 1)
qc.cx(1, 2)
data = np.array([0.2, 0.6, 0.8])
weights = np.array([0.1, -0.2, 0.3])
values = dict(zip(x, data))
values.update(dict(zip(theta, weights)))
bound = qc.assign_parameters(values)
state = Statevector.from_instruction(bound)
print('실제 회전각:', np.round(np.pi * data + weights, 6))
print('남은 미지정 파라미터:', bound.num_parameters)
print('확률:', np.round(state.probabilities(), 6))
assert bound.num_parameters == 0
assert np.isclose(state.probabilities().sum(), 1.0)

Qiskit: 2.2.3
실제 회전각: [0.728319 1.684956 2.813274]
남은 미지정 파라미터: 0
확률: [0.010331 0.001887 0.473317 0.0547   0.376511 0.068764 0.012988 0.001501]


입력과 학습 파라미터가 모두 결합되어 미지정 파라미터가 0개입니다. 최적화기를 연결할 때는 weights를 갱신하면서 목적값을 계산합니다.

In [13]:
import numpy as np
from importlib.metadata import version
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('Qiskit:', version('qiskit'))
data = np.array([0.9, 0.1, 0.5])
scale = 0.8
angles = scale * np.pi * data
qc = QuantumCircuit(3)
for i, angle in enumerate(angles):
    qc.ry(float(angle), i)
state = Statevector.from_instruction(qc)
p1 = np.array([state.probabilities([i])[1] for i in range(3)])
print('스케일:', scale)
print('각도:', np.round(angles, 6))
print('각 큐비트의 1 확률:', np.round(p1, 6))
assert np.allclose(p1, np.sin(angles / 2)**2)

Qiskit: 2.2.3
스케일: 0.8
각도: [2.261947 0.251327 1.256637]
각 큐비트의 1 확률: [0.818712 0.015708 0.345492]


스케일을 바꾸면 입력 간 거리와 측정 확률이 달라집니다. 적합한 값은 학습·검증 절차에서 선택하고 시험 데이터에는 확정된 값을 적용합니다.